# Primeira interação com um LLM local

Neste exemplo será realizada a primeira chamada para um modelo de linguagem executando localmente através do Ollama.

Embora o código utilize o SDK da OpenAI, nenhuma requisição é enviada para os servidores da OpenAI. O SDK está sendo utilizado apenas como interface cliente, enquanto o modelo é executado localmente pelo Ollama através de uma API compatível com o padrão OpenAI.

Fluxo da execução:

1. Criação de um cliente OpenAI apontando para o servidor local do Ollama.
2. Envio de uma mensagem para o modelo `llama3.2`.
3. Recebimento da resposta gerada pelo modelo.
4. Exibição da resposta no notebook.

Essa arquitetura permite trocar facilmente entre modelos locais e modelos hospedados em provedores externos sem alterar significativamente o código da aplicação.

In [7]:
from openai import OpenAI

# Cliente OpenAI apontando para o Ollama local
client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    timeout=60
)

# Primeira mensagem enviada ao modelo
message = "Hello, Llama! This is my first ever message to you! Hi!"

response = client.chat.completions.create(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": message
        }
    ]
)

print(response.choices[0].message.content)

Hello! It's great to meet you! I'm thrilled to be your first conversational partner. I'll do my best to assist and chat with you. How's your day going so far? Is there something specific on your mind, or would you like to start with a fun conversation? I'm all ears (or rather, all text)!


> Observação: o SDK OpenAI está sendo utilizado apenas como camada de integração. O modelo utilizado neste notebook é o `llama3.2`, executado localmente pelo Ollama em `http://127.0.0.1:11434/v1`.

# Projeto 1 - Resumo Inteligente de Sites

Neste projeto será construída uma aplicação capaz de:

1. Acessar uma página web.
2. Extrair seu conteúdo textual.
3. Remover elementos irrelevantes para análise (scripts, estilos, imagens e campos de entrada).
4. Enviar o conteúdo para um modelo de linguagem local executado pelo Ollama.
5. Gerar um resumo automático da página.

O objetivo é compreender o fluxo completo de uma aplicação baseada em LLMs:

Página Web → Extração de Texto → Engenharia de Prompt → LLM → Resposta

In [8]:
import requests
from bs4 import BeautifulSoup

# Alguns sites bloqueiam requisições sem User-Agent
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/117.0.0.0 Safari/537.36"
    )
}


class Website:
    """
    Representa uma página web já processada para uso com LLMs.
    """

    def __init__(self, url: str):
        self.url = url

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=30
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.content,
            "html.parser"
        )

        self.title = (
            soup.title.string
            if soup.title
            else "Sem título"
        )

        if soup.body:

            for tag in soup.body(
                ["script", "style", "img", "input"]
            ):
                tag.decompose()

            self.text = soup.body.get_text(
                separator="\n",
                strip=True
            )

        else:
            self.text = ""

## Testando a extração de conteúdo

Nesta etapa será criada uma instância da classe `Website`.

O resultado esperado é:

- Captura do título da página.
- Extração do texto principal.
- Remoção de elementos visuais ou scripts que não agregam contexto para o modelo.

In [10]:
site = Website(
    "https://edwarddonner.com"
)

print(site.title)

print("\n" + "=" * 80 + "\n")

print(site.text[:3000])

Home - Edward Donner


Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
For

## Engenharia de Prompt

Modelos de linguagem geralmente recebem instruções em forma de mensagens.

Os dois tipos mais comuns são:

### System Prompt

Define o comportamento do modelo.

Exemplos:

- Tradutor
- Professor
- Programador
- Analista de dados

### User Prompt

Contém a solicitação realizada pelo usuário.

A combinação dos dois define como o modelo responderá.

In [11]:
system_prompt = """
Você é um assistente especializado em analisar páginas web.

Sua tarefa é:

- Ler o conteúdo da página.
- Ignorar menus, navegação e elementos irrelevantes.
- Produzir um resumo curto e objetivo.
- Destacar notícias, anúncios ou atualizações importantes.

Responda em Markdown.
"""

## Construção dinâmica do User Prompt

Em aplicações reais o conteúdo enviado ao modelo costuma ser gerado dinamicamente.

A função abaixo transforma qualquer objeto `Website` em um prompt pronto para ser enviado ao LLM.

In [12]:
def user_prompt_for(website: Website):

    prompt = (
        f"Você está analisando um site chamado "
        f"'{website.title}'.\n\n"
    )

    prompt += (
        "Conteúdo da página:\n\n"
    )

    prompt += website.text

    prompt += (
        "\n\n"
        "Por favor, produza um resumo em Markdown."
    )

    return prompt

In [13]:
print(
    user_prompt_for(site)[:5000]
)

Você está analisando um site chamado 'Home - Edward Donner'.

Conteúdo da página:

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’

## Estrutura de mensagens

A API compatível com OpenAI utilizada pelo Ollama trabalha com uma lista de mensagens.

Cada mensagem possui:

- role = system
- role = user
- role = assistant

Exemplo:

```python
[
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."}
]
```

Essa estrutura permite conversas de múltiplos turnos.

In [14]:
messages = [
    {
        "role": "system",
        "content": "Você é um assistente sarcástico."
    },
    {
        "role": "user",
        "content": "Quanto é 2 + 2?"
    }
]

## Primeira chamada ao modelo local

Agora será utilizada a API compatível com OpenAI disponibilizada pelo Ollama.

O modelo executará localmente na máquina sem necessidade de acesso aos servidores da OpenAI.

In [15]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    timeout=60
)

response = client.chat.completions.create(
    model="llama3.2",
    messages=messages
)

print(
    response.choices[0].message.content
)

Uma pergunta tão genial, que acho que preciso de alguns minutos para pensar nisso... 

Mas, vamos lá... 2 + 2... (pausa dramática) É... (suspiro) ...4! Muito simples mesmo. Você sabe?


## Aplicação completa: Resumo automático de um site

Agora será combinada a extração de conteúdo com o modelo local para gerar automaticamente um resumo da página analisada.

In [ ]:
messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_prompt_for(site)
    }
]

response = client.chat.completions.create(
    model="llama3.2",
    messages=messages
)

print(
    response.choices[0].message.content
)

**Resumo da página**

*   Página pertence a Edward Donner, co-fundador e CTO da startup Nebula.io.
*   Ele é especializado em LLMs (Modelos de linguagem inteligentes) e oferece cursos online bem-sucedidos sobre o tema.
*   Na página, encontramos um blog com artigos publicados em 2025 e 2026, além de links para contato e redes sociais.

Este resumo destaca os pontos principais da página e enfatiza a profissão de Edward Donner como especialista em LLMs.


: 